# Create Sample Test CSVs

Creates sample CSV files with on-chain features only
Off-chain features (Reddit, Market) are merged by the backend

In [1]:
import pandas as pd
from pathlib import Path

## Configuration

In [9]:
TRAIN_DATA_PATH = '../data/processed/final/final_train_data.parquet'
OUTPUT_DIR = Path('../data/sample_batches')
NUM_SAMPLES = 5
MIN_ADDRESSES = 20

ONCHAIN_FEATURES = ['normal_total_cnt', 'uniq_peers_cnt', 'burst_max_tx_5m', 'normal_sent_cnt']

## Load Training Data

In [10]:
train = pd.read_parquet(TRAIN_DATA_PATH)
train['address'] = train['address'].str.lower()

print(f"Loaded {len(train):,} records")
print(f"Date range: {train['day'].min()} to {train['day'].max()}")

Loaded 59,575 records
Date range: 2017-03-15 to 2021-11-01


## Select Days

In [11]:
label_col = 'is_anomalous'

daily_stats = train.groupby('day').agg({
    'address': 'nunique',
    label_col: 'sum'
}).reset_index()
daily_stats.columns = ['day', 'unique_addrs', 'anomaly_count']

good_days = daily_stats[daily_stats['unique_addrs'] >= MIN_ADDRESSES].sort_values('unique_addrs', ascending=False)
selected_days = good_days.head(NUM_SAMPLES)['day'].tolist()

print(f"Selected {len(selected_days)} days:")
for d in selected_days:
    row = daily_stats[daily_stats['day'] == d].iloc[0]
    print(f"  {d}: {row['unique_addrs']} addresses, {row['anomaly_count']} anomalies")

Selected 5 days:
  2021-10-06: 260 addresses, 141 anomalies
  2021-10-12: 260 addresses, 130 anomalies
  2021-10-13: 256 addresses, 136 anomalies
  2021-10-08: 253 addresses, 120 anomalies
  2021-10-15: 252 addresses, 143 anomalies


## Create CSV Files

In [12]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for f in OUTPUT_DIR.glob('*.csv'):
    f.unlink()

for day in selected_days:
    day_data = train[train['day'] == day].copy()
    day_str = str(day).replace("-", "_")

    sample = day_data[['address'] + ONCHAIN_FEATURES].copy()
    sample['date'] = str(day)
    sample['Class'] = day_data[label_col].values
    
    cols = ['address', 'date'] + ONCHAIN_FEATURES + ['Class']
    sample = sample[cols]
    
    # Labeled version
    sample.to_csv(OUTPUT_DIR / f'labeled_{day_str}.csv', index=False)
    
    # Unlabeled version
    sample.drop(columns=['Class']).to_csv(OUTPUT_DIR / f'unlabeled_{day_str}.csv', index=False)
    
    print(f"{day}: {len(sample)} addresses, {int(sample['Class'].sum())} anomalies")

print("\nDone! Files contain ON-CHAIN features only.")

2021-10-06: 260 addresses, 141 anomalies
2021-10-12: 260 addresses, 130 anomalies
2021-10-13: 256 addresses, 136 anomalies
2021-10-08: 253 addresses, 120 anomalies
2021-10-15: 252 addresses, 143 anomalies

Done! Files contain ON-CHAIN features only.
